<a href="https://colab.research.google.com/github/Rahul-Hadiyal/Elevate/blob/feat%2Fcolab-training/colab-training/Elevate_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Elevate: Amazon ML Challenge — Business Entity Resolution
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rahul-Hadiyal/Elevate/blob/feat/colab-training/Elevate_Colab_Training.ipynb)

## 🌙 Master Overnight End-to-End Pipeline (Phases 1–8)

This notebook executes the entire **Gemini Implementation Plan** autonomously overnight and saves your final, verified competition submission directly to Google Drive:
- **Phase 1**: Blocking Recall sweep & Miss Classification (Gate 1)
- **Phase 2**: Uncapped candidate generation (`cap=50`, `max_cands_per_source=999`, vetoes disabled)
- **Phase 3**: 5-Fold split & 5-Seed Ensemble LightGBM training (`scale_pos_weight=1.0`)
- **Phase 4**: Platt Sigmoid Probability Calibration & ECE verification (Gate 4)
- **Phase 5**: Exact Expected-$F_{0.5}$ Poisson-Binomial Decision Engine Optimization (Gate 5)
- **Phase 7**: Full official test inference on all 1,732,544 records, generating `candidate_pairs.tsv` & `matching_results.tsv`
- **Phase 8**: 8 Submission Invariant validations & packaging `Elevate_final_submission.zip`

### Step 1: Optional Google Drive Mount (Saves Your Finished Files Overnight)

In [ ]:
from google.colab import drive
import os

# Mount Drive so finished submissions are saved permanently even if Colab disconnects
drive.mount('/content/drive')
drive_save_dir = "/content/drive/MyDrive/Elevate_Submission"
os.makedirs(drive_save_dir, exist_ok=True)
print("Google Drive connected! Results will be saved to:", drive_save_dir)

### Step 2: Clone Repository and Pull Dataset

In [ ]:
# Install git-lfs
!apt-get update -qq && apt-get install -y git-lfs
!git lfs install

# Clone the Elevate repository on branch feat/colab-training
!git clone -b feat/colab-training https://github.com/Rahul-Hadiyal/Elevate.git
%cd /content/Elevate

# Pull large dataset files via Git LFS
!git lfs pull

# Unzip dataset if packed in student_resource.zip
import os, glob, zipfile
zip_files = glob.glob("*student_resource*.zip")
if zip_files and not os.path.exists("dataset/train"):
    print(f"Extracting {zip_files[0]}...")
    with zipfile.ZipFile(zip_files[0], 'r') as z:
        z.extractall(".")
print("Dataset verified ready:", os.path.exists("dataset/train/train_source1.tsv"))

### Step 3: Install Dependencies & Run Verification Suite

In [ ]:
%cd /content/Elevate/code/business_entity_resolution
!pip install -q -r requirements.txt rapidfuzz lightgbm xgboost
!python -m pytest tests/ -v --tb=short

### Step 4: Run Master Overnight Pipeline (Phases 1 through 8)

In [ ]:
# Runs the complete pipeline, sweeps thresholds, performs test inference, checks 8 invariants, and packages submission zip
!python src/run_overnight_master.py

### Step 5: Save Finished Product to Google Drive & Local Browser

In [ ]:
import shutil
from google.colab import files

submission_zip = "/content/Elevate/Elevate_final_submission.zip"
drive_target = "/content/drive/MyDrive/Elevate_Submission/Elevate_final_submission.zip"

if os.path.exists(submission_zip):
    print(f"Copying final submission package to Google Drive ({drive_target})...")
    shutil.copyfile(submission_zip, drive_target)
    shutil.copyfile("/content/Elevate/output/matching_results.tsv", "/content/drive/MyDrive/Elevate_Submission/matching_results.tsv")
    shutil.copyfile("/content/Elevate/output/candidate_pairs.tsv", "/content/drive/MyDrive/Elevate_Submission/candidate_pairs.tsv")
    print("SUCCESS! Your finished product is permanently saved in your Google Drive.")
    print("Triggering browser download...")
    files.download(submission_zip)
else:
    print("Pipeline output files:", os.listdir("/content/Elevate/output") if os.path.exists("/content/Elevate/output") else "None")